In [2]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.4/184.4 MB 10.2 MB/s  0:00:18m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.8 MB/s  0:00:00
Using cached fsspec-2025.12.0-py3-none-any.whl (201 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 10.7 MB/s  0:00:00 eta 0:00:01
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached filelock-3.20.0-py3-none-any.whl (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 10.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [torchaudio]2 [torchvision]
Note: you may need to restart the kernel to use updated packages.

In [4]:
import torch
import torch.nn as nn
from torchvision import models

In [ ]:

# 1. Load the pre-trained MobileNetV2
model = models.mobilenet_v2(weights='DEFAULT')

# 2. Modify the first layer to accept grayscale (1 channel)
# Original: Conv2d(3, 32, kernel_size=(3, 3)...)
existing_layer = model.features[0][0]
model.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), 
                                 stride=(2, 2), padding=(1, 1), bias=False)

# 3. Freeze all parameters (prevent gradients from updating base weights)
for param in model.parameters():
    param.requires_grad = False

# 4. Replace the classifier "Head" 
# MobileNetV2's classifier is a Dropout + Linear layer
num_ftrs = model.last_channel # 1280
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(num_ftrs, 10) # 10 classes for digits 0-9
)

In [19]:
import torch.optim as optim

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loss Function: CrossEntropy is standard for multi-class (0-9 digits)
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam is a safe, fast-learning default
# We only pass parameters where requires_grad=True
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

In [15]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# MobileNet expects specific normalization values if using pre-trained weights
transform = transforms.Compose([
    transforms.Resize((128, 128)),     # Upscale 28x28 to 128x128
    transforms.ToTensor(),             # Convert to [0, 1] range
    transforms.Normalize((0.1307,), (0.3081,)) # Standard MNIST mean/std
])

In [25]:
# 1. Load the "Big" training set
full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# 2. Define the sizes (e.g., 50,000 for training, 10,000 for validation)
train_size = 50000
val_size = 10000

# 3. Create the two separate dataset objects
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# 4. Now create your DataLoaders
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, epochs=5):
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            # 1. Move data to the same device as the model
            images = images.to(device)
            labels = labels.to(device)

            # 2. Forward pass: Get predictions
            outputs = model(images)
            
            # 3. Calculate Loss
            loss = criterion(outputs, labels)

            # 4. Backward pass: Calculate gradients and update
            optimizer.zero_grad() # Clear previous gradients
            loss.backward()       # Compute new gradients
            optimizer.step()      # Update weights

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad(): # Disable gradient calculation for speed/memory
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1} | Train Loss: {running_loss/len(train_loader):.4f} | "
              f"Val Acc: {100 * correct / len(val_loader.dataset):.2f}%")



In [ ]:
train_model(model, train_loader, val_loader, optimizer, epochs=5)

In [28]:
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Final Test Accuracy: {100 * correct / total:.2f}%')
test_model(model, test_loader)

Final Test Accuracy: 91.39%


In [32]:
# Example: Fine-tuning everything (Stage 2)
for param in model.parameters():
    param.requires_grad = True

# Use a 10x or 100x smaller learning rate for the base
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)



In [33]:
train_model(model, train_loader, val_loader, optimizer, epochs=5)


Epoch 1 | Train Loss: 0.0000 | Val Acc: 95.63%
Epoch 2 | Train Loss: 0.0000 | Val Acc: 96.89%
Epoch 3 | Train Loss: 0.0000 | Val Acc: 97.31%
Epoch 4 | Train Loss: 0.0000 | Val Acc: 97.63%
Epoch 5 | Train Loss: 0.0000 | Val Acc: 98.07%


In [34]:
test_model(model, test_loader)

Final Test Accuracy: 97.74%


In [36]:
# Define the file path (use .pth or .pt extension)
PATH = "mobilenet_mnist.pth"

# Save the weights
torch.save(model.state_dict(), PATH)
print("Model weights saved!")

Model weights saved!
